# 🧮 How the improved system calculates its results

This notebook explains what happens inside the **improved (condition-aware)** system

---

## 🤖 formulas and rules

The improved system uses **no AI**: no ChatGPT, no language model, no trained machine-learning model. It consists only of **formulas and fixed rules**, so every result can be recalculated by hand.

| Component | Technique |  |
|---|---|---|
| Text search | BM25, a statistical formula (library `rank_bm25`) |  |
| Recognising places despite typos | String similarity (library `rapidfuzz`) |  |
| Dates such as *"20 Oct"* | Date parser (library `dateparser`) |  |
| Skill level, time, regions | Fixed word lists and rules |  |
| Scoring the conditions | Formulas using the thresholds in `surfability_thresholds.csv` |  |
| Explanation sentence | Text template filled with the computed numbers |  |
| *Optional:* dense search | Small embedding model `bge-small-en-v1.5` | switched off by default |



## 1️⃣ Understanding the query (rules)

- **Place:** the query is split into word groups of 1–3 words. Each group is compared with all place and spot names; a similarity of **at least 88 %** counts as a match, so typos such as *"Kaohsuing"* still work.
- **"near X":** all spots within **40 km** (straight-line distance, haversine formula) remain. Region phrases such as *"east coast"* keep the spots in those regions.
- **Skill level and time:** pattern matching in the text, e.g. *"tomorrow"* = day + 1, *"morning"* = 06:00–11:00, counted from the moment the snapshot was taken.

## 2️⃣ Text score (BM25)

For every word $t$ in the query, each spot description $d$ earns points:

$$
\text{points}(t, d) = \text{IDF}(t) \cdot \frac{f_{t,d} \cdot (k_1 + 1)}{f_{t,d} + k_1 \cdot \left(1 - b + b \cdot \frac{|d|}{\text{avg}|d|}\right)}
\qquad
\text{IDF}(t) = \log\left(1 + \frac{N - n_t + 0.5}{n_t + 0.5}\right)
$$

| Symbol | Meaning | Value |
|---|---|---|
| $f_{t,d}$ | how often word $t$ appears in description $d$ | – |
| $n_t$ | number of spots whose description contains $t$ | – |
| $N$ | number of spots | 35 |
| $\lvert d\rvert$ / $\text{avg}\lvert d\rvert$ | description length / average length (length correction) | – |
| $k_1$ | how quickly repeated words stop adding points | 1.2 |
| $b$ | strength of the length correction | 0.75 |

**Intuition:**
- **Rare words count more:** *reef* earns more than *break*, which appears almost everywhere.
- **Repeated words** earn more points, but with diminishing returns.
- **Long descriptions** are scored down slightly.

**Slang expansions** (e.g. *newbie* → *beginner*) are added with **weight 0.5**. The sum is divided by the highest score of all spots, so the **text score lies between 0 and 1**.

## 3️⃣ Condition fit (scoring the conditions)

For **every hour** in the time window, six factors between 0 and 1 are calculated and **multiplied**:

$$
CF = \text{size} \times \text{period} \times \text{direction} \times \text{wind} \times \text{tide}^{0.5} \times \text{level}
$$

The **best hour** in the time window counts. Multiplying means a single knock-out factor makes the score low: perfect wind does not help when there are no waves.

### Worked example (illustration)

**Wai'ao, intermediate surfer.** Forecast: 0.8 m swell from the east, 5 s period, 7 km/h wind from the southeast, mid tide.

| Factor | Rule (thresholds for intermediate) | Calculation | Value |
|---|---|---|---|
| 🌊 **Size** | 1 from 1.0 to 1.5 m; outside that it drops in 0.1 steps, reaching 0.1 at 0.6 m and at 1.8 m (0 below 0.6 m, unsafe above 1.8 m) | 0.1 + 0.9 × (0.8 − 0.6) / (1.0 − 0.6) = 0.55 → rounded down to 0.1 steps | **0.50** |
| ⏱️ **Period** | 0.4 at 6 s or less, rises to 1 at 9 s | 5 s ≤ 6 s | **0.40** |
| 🧭 **Direction** | 1 if the swell is inside the spot's window (60–160°); outside it drops in 0.1 steps, never below 0.1 | east = 90°, inside | **1.00** |
| 💨 **Wind** | The beach faces 98°, so offshore wind would come from 278°. Southeast (135°) is onshore. Penalty = onshore share × wind strength | 1 − 0.87 × (7 − 5.5) / (28 − 5.5) | **0.94** |
| 🌙 **Tide** | 1 if the tide stage matches (Wai'ao: mid/high) | mid matches | **1.00** |
| 🏄 **Level** | 1 if the spot profile lists this skill level | intermediate is listed | **1.00** |

$$
CF = 0.50 \times 0.40 \times 1.00 \times 0.94 \times 1.00 \times 1.00 \approx 0.19 \;\Rightarrow\; \text{"Poor", explanation: "Limited by short period"}
$$

- The **smallest factor** (here the period) is named as the reason in the explanation.
- From **0.5** a spot counts as **surfable ("Fair")**, from **0.75** as **"Good"**.

### Correction by buoy observations

If a nearby buoy measured within the **last 3 hours** and the measurement differs from the forecast by **more than 30 %**, the forecast wave height for the **next 12 hours** is adjusted step by step towards the measurement. The explanation mentions this.

## 4️⃣ Final score

$$
\text{score} = \alpha \cdot \text{text score} + (1 - \alpha) \cdot CF
$$

| Query type | $\alpha$ | Meaning |
|---|---|---|
| Asks about conditions, time or skill (default) | **0.5** | text and conditions count equally |
| No conditions angle, e.g. *"reef break with showers"* | **0.85** | text counts more |

**Safety rule:** if a spot is unsafe for the user's level (waves too big, or the spot is not meant for that level), **1 point is subtracted**. It drops to the bottom of the list and gets a **warning**.

> 🎛️ **Tuning:** with `TUNE_ALPHA = True` in notebook 05, $\alpha$ is optimised on the dev queries. This is the only place where anything is "learned" from data, and even here the system simply tries the fixed values 0.0, 0.1, …, 1.0 and keeps the best one.

## 5️⃣ The thresholds file: `data/resources/surfability_thresholds.csv`

All numbers used by the condition-fit score live in **one editable table**, not in the code. Each row is one parameter, with one value per skill level, a unit, a description and a **source**. Rows without an external source are marked **DESIGN CHOICE** and give a reason instead.

> ✏️ **To change a value:** edit the CSV (e.g. in Excel), save, and restart the demo (or re-run `03_surfability` / `04_search`). No code changes are needed. Record the new source in the `source` column.

### 🌊 Wave size: the size score

The size score works in **0.1 steps**: it is **1** inside the ideal range (`wave_ideal_low_m` to `wave_ideal_high_m`). Below the range it drops step by step (0.9, 0.8 …) to **0.1** at `wave_min_m`; above the range it drops to **0.1** at `wave_max_m`. It is **0** below `wave_min_m` (flat) and above `wave_max_m` (unsafe). The steps are evenly spaced between the ideal limit and the outer limit, and the score is rounded down to the nearest 0.1.

| Parameter | Beginner | Intermediate | Advanced | Meaning |
|---|---|---|---|---|
| `wave_min_m` | 0.1 m* | 0.6 m | 1.0 m | Lower limit: score 0.1 here, 0 below (flat) |
| `wave_ideal_low_m` | 0.3 m | 1.0 m | 1.5 m | Ideal range starts (score 1) |
| `wave_ideal_high_m` | 0.6 m | 1.5 m | 3.0 m* | Ideal range ends (score 1 up to here) |
| `wave_max_m` | 1.0 m | 1.8 m | 4.0 m* | Upper limit: score 0.1 here; above it **unsafe** (score 0 and a safety warning) |

**Sources:**
- Beginner range 1–2 ft (0.3–0.6 m) is ideal, and 1 ft is about the smallest surfable size: Sharp Eye Surfboards.
- Ranges 1–3 ft beginner, 3–5 ft intermediate, 5 ft+ advanced: Climeradar.
- 6–8 ft is the line between intermediate and advanced: AlbumSurf.
- \* DESIGN CHOICES: the advanced upper values (the sources give no upper bound), and the beginner minimum of 0.1 m, which gives beginners graded scores below the 1 ft ideal minimum (very small waves are still rideable on a longboard, Sharp Eye).

> ⚠️ **Note:** the forecast gives the *offshore significant wave height*. The waves surfed at the beach are typically 50–100 % of the forecast swell height (Rapture Camps), so these thresholds are slightly cautious.

### ⏱️ Period: the period score

The period score is `period_floor_fit` at or below `period_min_s`, rising linearly to 1 at `period_ideal_s`.

| Parameter | Beginner | Intermediate | Advanced | Meaning |
|---|---|---|---|---|
| `period_min_s` | 5 s | 6 s | 7 s | At or below: short-period, choppy wind swell |
| `period_ideal_s` | 8 s | 9 s | 10 s | Score reaches 1: organised swell |
| `period_floor_fit` | 0.5 | 0.4 | 0.3 | Score for very short-period swell (DESIGN CHOICE: beginners in white water are least affected) |

**Sources:**
- 5–8 s is wind swell and 9–12 s medium-period swell: Rapture Camps.
- 5–9 s short and 10–13 s organised: Climeradar.
- 8–9 s still gives fun surfable waves: Foam Magazine.

### 🧭 Swell direction

| Parameter | All levels | Meaning |
|---|---|---|
| `swell_window_margin_deg` | 30° | Outside the spot's swell window, the direction score drops in 0.1 steps over this many degrees |
| `direction_floor_fit` | 0.1 | Lowest direction score; it is never 0 |

Both are DESIGN CHOICES: swell refracts and wraps around headlands, so energy from just outside a spot's window still arrives, only weaker. 30° is a bit more than one compass sector (22.5°), so the score drops 0.1 about every 3°. The minimum of 0.1 means a wrong direction cuts the condition fit to a tenth but does not set it to 0 on its own.

### 💨 Wind

| Parameter | Beginner | Intermediate | Advanced | Meaning |
|---|---|---|---|---|
| `wind_glassy_max_kmh` | 5.5 | 5.5 | 5.5 | Up to this speed the wind counts as light/glassy, whatever its direction |
| `wind_offshore_max_kmh` | 18.5 | 28 | 28 | Offshore wind up to this speed is ideal |
| `wind_offshore_limit_kmh` | 37 | 46 | 46 | Offshore wind this strong scores 0 (too hard to paddle into waves) |
| `wind_onshore_max_kmh` | 28 | 28 | 28 | Straight onshore wind this strong scores 0 |

**Sources:**
- Glassy = Beaufort force 0–1, up to 3 knots (Met Office).
- Beginners have little difficulty below 10 knots, while 10–20 knots makes balancing harder (Skateboarders HQ).
- 5–15 knots offshore is the sweet spot, 25+ knots offshore makes paddling difficult, and 15+ knots onshore badly degrades waves (Rapture Camps).

With these settings, a 10-knot (18.5 km/h) onshore wind scores about 0.4.

### 🌙 Tide and 🏄 spot level

| Parameter | Beginner | Intermediate | Advanced | Meaning |
|---|---|---|---|---|
| `tide_mismatch_fit` | 0.6 | 0.6 | 0.6 | Tide score when the tide stage is not one the spot prefers |
| `spot_skill_mismatch_fit` | **0.0** | 0.4 | 0.7 | Score when the spot profile does not list your skill level |

Both are DESIGN CHOICES:
- **Tide:** a wrong tide rarely makes a spot unsurfable, so it lowers the score instead of zeroing it.
- **Spot level (safety):** a spot whose profile excludes beginners is **never** recommended to a beginner (0.0). Intermediate and advanced surfers are penalised less.

### ⚖️ Weights: how decisive each factor is

The weights are the **exponents** in the product $CF = \text{size}^{w} \times \text{period}^{w} \times \dots$. A weight of 1 means normal influence, 0.5 means half as decisive.

| Parameter | Value | Reason (DESIGN CHOICE) |
|---|---|---|
| `weight_size`, `weight_period`, `weight_direction`, `weight_wind` | 1 | Each of these can make a spot unsurfable on its own |
| `weight_tide` | 0.5 | The tide stage comes from a coarse offshore sea-level model, and tide preferences are the least well-sourced field in the spot profiles |

### ✅ Verdict thresholds and ☀️ daylight

| Parameter | Value | Meaning |
|---|---|---|
| `surfable_threshold` | 0.5 | From this condition fit a spot counts as **surfable ("Fair")**; also used for the surfable / not-surfable evaluation |
| `good_threshold` | 0.75 | From this condition fit the verdict is **"Good"** (affects the wording only) |
| `first_light_hour` | 06:00 | First hour considered |
| `last_light_hour` | 18:00 | Hours from this one on are not considered |

- **Verdict thresholds:** DESIGN CHOICES. 0.5 is the midpoint of the score scale; tune it on the evaluation dev queries and report how sensitive the results are.
- **Daylight hours:** in 2025, sunrise in Taiwan ranged from 05:02 (June) to 06:40 (January) and sunset from 17:02 (December) to 18:47 (June). These figures were computed with the Open-Meteo Archive API for Keelung and Kenting; 06:00–18:00 is a year-round compromise.

> 💡 **Verdict vs. score:** the verdict (Good / Fair / Poor) uses **only** the condition fit and these thresholds. The ranking score also includes the text match (`0.5 × text + 0.5 × CF`). A spot can therefore rank high while its verdict is "Poor", because its description matches the query well even though the waves are not good.

### 📚 Sources cited in the table

| Source | Used for |
|---|---|
| [Sharp Eye Surfboards FAQ](https://sharpeyesurfboards.com/pages/faqs) | Beginner and intermediate wave sizes |
| [Climeradar: Surfing weather cheat sheet](https://climeradar.com/blog/surfing-weather-cheat-sheet) | Wave-size ranges per level, period categories |
| [AlbumSurf: 6–8 ft surf](https://node.albumsurf.com/ride-the-wave-understanding-surf-conditions-between-6-and-8-feet) | Intermediate/advanced boundary |
| [Rapture Camps: Surf forecast basics](https://www.rapturecamps.com/learn-to-surf/wave-knowledge/surf-forecast-basics) | Period categories, face vs. forecast height |
| [Rapture Camps: Tides and wind](https://www.rapturecamps.com/learn-to-surf/wave-knowledge/tides-and-wind) | Wind speeds, tide behaviour |
| [Foam Magazine: Wave period](https://www.foammagazine.com/wave-period/) | 8–9 s still surfable |
| [Skateboarders HQ: Wind speed for surfing](https://skateboardershq.com/what-wind-speed-surfing/) | Beginner wind limits |
| [Met Office: Beaufort scale](https://weather.metoffice.gov.uk/guides/coast-and-sea/beaufort-scale) | Glassy / light wind |
| [Open-Meteo Historical Weather API](https://open-meteo.com/en/docs/historical-weather-api) | Sunrise and sunset times |

> ⚠️ These are surf-education websites, not peer-reviewed research. For the report, have a local surf school or instructor confirm the values and cite that as well.